In [2]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_openai import AzureChatOpenAI
import os

function = {
    "name": "create_quiz",
    "description": "function that takes a list of questions and answers and returns a quiz",
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {
                            "type": "string",
                        },
                        "answers": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "answer": {
                                        "type": "string",
                                    },
                                    "correct": {
                                        "type": "boolean",
                                    },
                                },
                                "required": ["answer", "correct"],
                            },
                        },
                    },
                    "required": ["question", "answers"],
                },
            }
        },
        "required": ["questions"],
    },
}
llm=AzureChatOpenAI(
    api_version=os.getenv("API_VERSION"),
    azure_endpoint=os.getenv("ENDPOINT"),
    azure_deployment=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("API_KEY"),
).bind(
    function_call={
        "name": "create_quiz",
    },
    functions=[
        function,
    ],
)

# llm = ChatOpenAI(
#     temperature=0.1,
# ).bind(
#     function_call={
#         "name": "create_quiz",
#     },
#     functions=[
#         function,
#     ],
# )

prompt = PromptTemplate.from_template("Make a quiz about {city}")

chain = prompt | llm

response = chain.invoke({"city": "rome"})


response = response.additional_kwargs["function_call"]["arguments"]

response

'{\n  "questions": [\n    {\n      "question": "What is the capital city of Rome?",\n      "answers": [\n        {\n          "answer": "Rome",\n          "correct": true\n        },\n        {\n          "answer": "Milan",\n          "correct": false\n        },\n        {\n          "answer": "Venice",\n          "correct": false\n        },\n        {\n          "answer": "Florence",\n          "correct": false\n        }\n      ]\n    },\n    {\n      "question": "Which famous ancient structure is located in Rome?",\n      "answers": [\n        {\n          "answer": "The Colosseum",\n          "correct": true\n        },\n        {\n          "answer": "The Eiffel Tower",\n          "correct": false\n        },\n        {\n          "answer": "The Great Wall of China",\n          "correct": false\n        },\n        {\n          "answer": "The Taj Mahal",\n          "correct": false\n        }\n      ]\n    },\n    {\n      "question": "Who was the first emperor of Rome?",\n     

In [3]:
import json

for question in json.loads(response)["questions"]:
    print(question)

{'question': 'What is the capital city of Rome?', 'answers': [{'answer': 'Rome', 'correct': True}, {'answer': 'Milan', 'correct': False}, {'answer': 'Venice', 'correct': False}, {'answer': 'Florence', 'correct': False}]}
{'question': 'Which famous ancient structure is located in Rome?', 'answers': [{'answer': 'The Colosseum', 'correct': True}, {'answer': 'The Eiffel Tower', 'correct': False}, {'answer': 'The Great Wall of China', 'correct': False}, {'answer': 'The Taj Mahal', 'correct': False}]}
{'question': 'Who was the first emperor of Rome?', 'answers': [{'answer': 'Julius Caesar', 'correct': False}, {'answer': 'Augustus', 'correct': True}, {'answer': 'Nero', 'correct': False}, {'answer': 'Constantine', 'correct': False}]}
{'question': 'Which famous artist painted the ceiling of the Sistine Chapel in Rome?', 'answers': [{'answer': 'Leonardo da Vinci', 'correct': False}, {'answer': 'Raphael', 'correct': False}, {'answer': 'Michelangelo', 'correct': True}, {'answer': 'Pablo Picasso', 